# Time-boxed DPO research run (Colab T4)

Run code cells from top to bottom. This notebook is pinned to GitHub commit da23a88af5821953b7668e05d2c0b15fa2f32d46 and stops if GitHub main no longer matches it. Before running, add HF_TOKEN as a write-enabled Colab Secret and enable notebook access. The token is retrieved with google.colab.userdata and is never printed or written into the repository. The private Hub repository is adhill/dpo-experiment-checkpoint.

Cell order: (1) fetch and verify the commit, retrieve the secret; (2) verify the T4 and install pinned Colab dependencies; (3) write a runtime-only copy of the existing scaled config and verify the experiment identity; (4) authenticate, upload, restore, compare, and delete a tiny temporary Hub artifact; (5) only after cell 4 succeeds, start or resume all five configured arms.

The plan remains SFT on 512 configured examples, full-data DPO at beta 0.1 and 0.5, and 128-pair margin-selected and random DPO, seed 42. The runner uses one saved SFT checkpoint for every DPO arm, pins UltraFeedback, logs every optimizer step to CSV and JSONL, uploads checkpoints to Hub, and restores or resumes incomplete work. It then evaluates Base, SFT, and DPO on the configured validation split. It does not use the final-test partition.

The T4 timing smoke measured about 3.06 seconds per SFT step and 6.12 seconds per DPO step. These timings exclude full-run setup, checkpoint transfers, and validation generation; the three-hour estimate is not a guarantee. Do not run cell 5 until you are ready to start training.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

REPO_URL = "https://github.com/Adhilmohdkt/dpo-from-scratch.git"
BRANCH = "main"
EXPECTED_COMMIT = "da23a88af5821953b7668e05d2c0b15fa2f32d46"
PROJECT_DIR = Path("/content/dpo-from-scratch-da23a88")

if PROJECT_DIR.exists():
    if not (PROJECT_DIR / ".git").is_dir():
        raise RuntimeError(f"Project path exists but is not a Git checkout: {PROJECT_DIR}")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(PROJECT_DIR)], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)

remote_commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", f"origin/{BRANCH}"], text=True
).strip()
if remote_commit != EXPECTED_COMMIT:
    raise RuntimeError(
        f"GitHub {BRANCH} is {remote_commit}, expected {EXPECTED_COMMIT}. "
        "Review and update the notebook commit pin before continuing."
    )
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "--detach", EXPECTED_COMMIT], check=True)
checked_out_commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True
).strip()
if checked_out_commit != EXPECTED_COMMIT:
    raise RuntimeError("The project checkout does not match the reviewed commit.")

try:
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError(
        "Could not read HF_TOKEN from Colab Secrets. Confirm the secret exists and notebook access is enabled."
    ) from None
if not _hf_token:
    raise RuntimeError("HF_TOKEN is empty or unavailable in Colab Secrets.")
os.environ["HF_TOKEN"] = _hf_token
del _hf_token
%cd /content/dpo-from-scratch-da23a88
print({"github_main_commit": remote_commit, "checked_out_commit": checked_out_commit, "hf_token_available": True})

In [ ]:
import subprocess, sys, torch
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print({"python": sys.version.split()[0], "torch": torch.__version__, "cuda_available": torch.cuda.is_available(), "gpu": GPU_NAME})
if not torch.cuda.is_available() or "T4" not in GPU_NAME:
    raise RuntimeError("This run requires an active Tesla T4. Select a T4 accelerator and reconnect.")
managed_torch = torch.__version__
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
from importlib.metadata import version
if version("torch").split("+")[0] != managed_torch.split("+")[0]:
    raise RuntimeError("Dependency installation changed Colab-managed PyTorch; stop and reconnect to a fresh runtime.")
import transformers, peft, datasets, trl, accelerate, yaml
print({"transformers": transformers.__version__, "peft": peft.__version__, "datasets": datasets.__version__, "trl": trl.__version__, "accelerate": accelerate.__version__})

In [ ]:
import yaml

BASE_CONFIG_PATH = Path("configs/scaled_t4.yaml")
CONFIG_PATH = Path("/content/scaled_t4_hub_run.yaml")
HUB_REPO_ID = "adhill/dpo-experiment-checkpoint"
config = yaml.safe_load(BASE_CONFIG_PATH.read_text(encoding="utf-8"))
expected = {
    "seed": 42,
    "training_pairs": 512,
    "betas": [0.1, 0.5],
    "selection_pairs": 128,
    "selection_beta": 0.1,
    "dataset_revision": "3949bf5f8c17c394422ccfab0c31ea9c20bdeb85",
}
actual = {
    "seed": config["seed"],
    "training_pairs": config["data"]["max_train_samples"],
    "betas": config["experiment"]["beta_values"],
    "selection_pairs": config["experiment"]["selection_subset_size"],
    "selection_beta": config["experiment"]["selection_beta"],
    "dataset_revision": config["data"]["revision"],
}
if actual != expected or config["experiment"]["seeds"] != [42]:
    raise RuntimeError(f"Scaled experiment config differs from the reviewed plan: {actual}")
config["hub"]["repo_id"] = HUB_REPO_ID
if config["hub"].get("private") is not True or config["hub"].get("enabled") is not True:
    raise RuntimeError("Hub persistence must remain enabled for a private repository.")
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print({"runtime_config": str(CONFIG_PATH), "plan": actual, "seeds": config["experiment"]["seeds"], "private_hub_repo": HUB_REPO_ID})

In [ ]:
# Mandatory gate: failure raises an error and the experiment cell remains blocked.
import hashlib, os, shutil, tempfile, uuid
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi
from src.hub import ensure_hub_repo, hub_run_path, push_run_snapshot, restore_run_snapshot

hub_settings = dict(config["hub"])
repo_id = "adhill/dpo-experiment-checkpoint"
run_id = uuid.uuid4().hex
hub_settings["run_prefix"] = f"{hub_settings['run_prefix']}-persistence-test"
test_root = Path(tempfile.mkdtemp(prefix=f"hf-hub-test-{run_id}-", dir="/content"))
source_dir = test_root / f"artifact-{run_id}"
restored_dir = test_root / "restored" / source_dir.name
hub_settings["restore_dir"] = str(test_root / "fresh-download-cache")
stage = "hub-persistence-smoke"
remote_file_path = None
artifact_may_exist = False
api = None
phase = "secret lookup"

try:
    _hf_token = userdata.get("HF_TOKEN")
    if not _hf_token:
        raise RuntimeError("HF_TOKEN is empty or unavailable")
    os.environ["HF_TOKEN"] = _hf_token
    api = HfApi(token=_hf_token)
    del _hf_token

    phase = "authentication"
    identity = api.whoami()
    if not isinstance(identity, dict) or not identity:
        raise RuntimeError("Hugging Face did not return an authenticated identity")
    repo_info = api.repo_info(repo_id=repo_id, repo_type="model")
    if getattr(repo_info, "private", None) is not True:
        raise RuntimeError("The target repository is not private")
    ensure_hub_repo(hub_settings)

    phase = "artifact upload"
    source_dir.mkdir(parents=True)
    payload = f"Colab Hub persistence smoke test {run_id}".encode("utf-8")
    (source_dir / "payload.txt").write_bytes(payload)
    remote_file_path = f"{hub_run_path(hub_settings, 42, stage, source_dir)}/payload.txt"
    artifact_may_exist = True
    uploaded_path = push_run_snapshot(source_dir, hub_settings, 42, stage)
    if uploaded_path != f"{repo_id}/{hub_run_path(hub_settings, 42, stage, source_dir)}":
        raise RuntimeError("Hub upload returned an unexpected artifact path")

    phase = "fresh-directory restore"
    if not restore_run_snapshot(restored_dir, hub_settings, 42, stage):
        raise RuntimeError("Hub restore reported that the uploaded artifact was not found")
    downloaded_file = restored_dir / "payload.txt"
    if not downloaded_file.is_file():
        raise RuntimeError("Restored test file is missing")

    phase = "content verification"
    downloaded_payload = downloaded_file.read_bytes()
    if downloaded_payload != payload or hashlib.sha256(downloaded_payload).digest() != hashlib.sha256(payload).digest():
        raise RuntimeError("Downloaded artifact contents differ from the uploaded contents")

    phase = "remote artifact deletion"
    api.delete_file(
        path_in_repo=remote_file_path,
        repo_id=repo_id,
        repo_type="model",
        commit_message="Remove temporary Colab Hub persistence test artifact",
    )
    if api.file_exists(repo_id=repo_id, filename=remote_file_path, repo_type="model"):
        raise RuntimeError("Hub still reports the temporary test file after deletion")
    artifact_may_exist = False
    HUB_PERSISTENCE_TEST_PASSED = True
    print({
        "authentication": "passed",
        "target_repo_private": True,
        "upload": "passed",
        "restore_to_fresh_directory": "passed",
        "content_match": True,
        "remote_cleanup": "passed",
        "artifact_sha256": hashlib.sha256(payload).hexdigest(),
    })
except Exception as exc:
    cleanup_note = ""
    if artifact_may_exist and api is not None and remote_file_path is not None:
        try:
            api.delete_file(
                path_in_repo=remote_file_path,
                repo_id=repo_id,
                repo_type="model",
                commit_message="Clean up failed temporary Colab Hub persistence test",
            )
            artifact_may_exist = False
        except Exception as cleanup_exc:
            cleanup_note = f" Temporary remote cleanup also failed ({type(cleanup_exc).__name__})."
    HUB_PERSISTENCE_TEST_PASSED = False
    raise RuntimeError(
        f"Hub persistence preflight failed during {phase} ({type(exc).__name__}). "
        f"Do not run the experiment cell.{cleanup_note}"
    ) from None
finally:
    try:
        del _hf_token
    except NameError:
        pass
    api = None
    shutil.rmtree(test_root, ignore_errors=True)

In [ ]:
if globals().get("HUB_PERSISTENCE_TEST_PASSED") is not True:
    raise RuntimeError("Hub upload/restore preflight has not passed in this runtime; training was not started.")

# This runs or resumes SFT, then both full-data beta arms and both 128-pair selection arms.
# The runner reuses one SFT checkpoint for all DPO arms and evaluates only on validation.
subprocess.run([sys.executable, "-m", "scripts.run_scaled_experiments", "--config", str(CONFIG_PATH)], check=True)
print("Experiment outputs:", Path(config["experiment"]["output_dir"]).resolve())